# GraphRAG Environment Setup

This notebook guides you through creating and configuring all Azure resources needed for the GraphRAG lab.

## Resources Created

| Resource | Description |
|----------|-------------|
| **Resource Group** | Container for all resources |
| **Cosmos DB (Gremlin API)** | Graph database for knowledge graph |
| **Azure AI Search** | Vector and text search service |
| **Azure OpenAI** | Embedding model for vectorization |
| **Storage Account** | Document storage (optional) |

## Estimated Costs

- Cosmos DB Gremlin (400 RU/s): ~$24/month
- Azure AI Search (Basic): ~$75/month
- Azure OpenAI (pay-per-use): ~$1-10/month
- **Total**: ~$100-110/month

⚠️ **Remember to run the cleanup cell at the end when done to avoid charges!**

---

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install azure-identity azure-mgmt-resource azure-mgmt-cosmosdb azure-mgmt-search azure-mgmt-cognitiveservices azure-mgmt-storage azure-mgmt-authorization gremlinpython azure-search-documents python-dotenv openai

---

## Step 2: Configure Environment Variables

Set your Azure configuration. You **must** provide your subscription ID.

In [ ]:
import os

# === REQUIRED: Set your Azure Subscription ID ===
os.environ['AZURE_SUBSCRIPTION_ID'] = ''  # e.g., '12345678-1234-1234-1234-123456789012'

# === OPTIONAL: Customize resource names ===
os.environ['RESOURCE_GROUP_NAME'] = 'rg-graphrag-lab'
os.environ['AZURE_LOCATION'] = 'eastus'  # Change if needed (check model availability)
os.environ['COSMOS_ACCOUNT_NAME'] = 'cosmos-graphrag-dev'
os.environ['COSMOS_DATABASE_NAME'] = 'graphrag-db'
os.environ['COSMOS_GRAPH_NAME'] = 'knowledge-graph'
os.environ['SEARCH_SERVICE_NAME'] = 'search-graphrag-dev'
os.environ['SEARCH_INDEX_NAME'] = 'chunks-index'
os.environ['OPENAI_ACCOUNT_NAME'] = 'oai-graphrag-dev'
os.environ['EMBEDDING_DEPLOYMENT_NAME'] = 'text-embedding-3-large'
os.environ['STORAGE_ACCOUNT_NAME'] = 'stgraphragdev'

# Validate subscription ID
if not os.environ.get('AZURE_SUBSCRIPTION_ID'):
    raise ValueError("⚠️ Please set AZURE_SUBSCRIPTION_ID above!")
else:
    print("✅ Configuration set!")
    print(f"   Subscription: {os.environ['AZURE_SUBSCRIPTION_ID'][:8]}...")
    print(f"   Resource Group: {os.environ['RESOURCE_GROUP_NAME']}")
    print(f"   Location: {os.environ['AZURE_LOCATION']}")

---

## Step 3: Authenticate with Azure

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

print("🔐 Authenticating with Azure...")

try:
    # Try DefaultAzureCredential first (works with az login, managed identity, etc.)
    credential = DefaultAzureCredential()
    # Test the credential
    from azure.mgmt.resource import ResourceManagementClient
    test_client = ResourceManagementClient(credential, os.environ['AZURE_SUBSCRIPTION_ID'])
    list(test_client.resource_groups.list())
    print("✅ Authenticated using DefaultAzureCredential")
except Exception as e:
    print(f"DefaultAzureCredential failed: {e}")
    print("\n🌐 Opening browser for interactive login...")
    credential = InteractiveBrowserCredential()
    print("✅ Authenticated using browser login")

---

## Step 4: Create Resource Group

In [ ]:
from azure.mgmt.resource import ResourceManagementClient

subscription_id = os.environ['AZURE_SUBSCRIPTION_ID']
resource_group = os.environ['RESOURCE_GROUP_NAME']
location = os.environ['AZURE_LOCATION']

print(f"📁 Creating Resource Group: {resource_group}")

resource_client = ResourceManagementClient(credential, subscription_id)

rg_result = resource_client.resource_groups.create_or_update(
    resource_group,
    {'location': location, 'tags': {'project': 'graphrag-lab', 'environment': 'dev'}}
)

print(f"✅ Resource Group created: {rg_result.name}")
print(f"   Location: {rg_result.location}")

---

## Step 5: Create Cosmos DB with Gremlin API

This creates:
- Cosmos DB account with Gremlin (Graph) API
- Database for the knowledge graph
- Graph container with partition key

In [ ]:
from azure.mgmt.cosmosdb import CosmosDBManagementClient
from azure.mgmt.cosmosdb.models import (
    DatabaseAccountCreateUpdateParameters,
    DatabaseAccountKind,
    Location,
    Capability,
    GremlinDatabaseCreateUpdateParameters,
    GremlinDatabaseResource,
    GremlinGraphCreateUpdateParameters,
    GremlinGraphResource,
    ContainerPartitionKey,
    CreateUpdateOptions
)

cosmos_account = os.environ['COSMOS_ACCOUNT_NAME']
cosmos_database = os.environ['COSMOS_DATABASE_NAME']
cosmos_graph = os.environ['COSMOS_GRAPH_NAME']

print(f"🌍 Creating Cosmos DB Account: {cosmos_account}")
print("   This may take 5-10 minutes...")

cosmos_client = CosmosDBManagementClient(credential, subscription_id)

# Create Cosmos DB account with Gremlin capability
account_params = DatabaseAccountCreateUpdateParameters(
    location=location,
    kind=DatabaseAccountKind.GLOBAL_DOCUMENT_DB,
    locations=[Location(location_name=location, failover_priority=0)],
    capabilities=[Capability(name='EnableGremlin')],
    tags={'project': 'graphrag-lab'}
)

poller = cosmos_client.database_accounts.begin_create_or_update(
    resource_group, cosmos_account, account_params
)
account_result = poller.result()
print(f"✅ Cosmos DB Account created: {account_result.name}")

In [ ]:
# Create Gremlin Database
print(f"📊 Creating Gremlin Database: {cosmos_database}")

db_params = GremlinDatabaseCreateUpdateParameters(
    resource=GremlinDatabaseResource(id=cosmos_database),
    options=CreateUpdateOptions()
)

poller = cosmos_client.gremlin_resources.begin_create_update_gremlin_database(
    resource_group, cosmos_account, cosmos_database, db_params
)
db_result = poller.result()
print(f"✅ Database created: {db_result.name}")

In [ ]:
# Create Gremlin Graph
print(f"📈 Creating Gremlin Graph: {cosmos_graph}")

graph_params = GremlinGraphCreateUpdateParameters(
    resource=GremlinGraphResource(
        id=cosmos_graph,
        partition_key=ContainerPartitionKey(paths=['/tenant'], kind='Hash')
    ),
    options=CreateUpdateOptions(throughput=400)  # 400 RU/s
)

poller = cosmos_client.gremlin_resources.begin_create_update_gremlin_graph(
    resource_group, cosmos_account, cosmos_database, cosmos_graph, graph_params
)
graph_result = poller.result()
print(f"✅ Graph created: {graph_result.name}")
print(f"   Partition Key: /tenant")
print(f"   Throughput: 400 RU/s")

---

## Step 6: Create Azure AI Search Service

In [ ]:
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.search.models import SearchService, Sku, Identity

search_service = os.environ['SEARCH_SERVICE_NAME']

print(f"🔍 Creating Azure AI Search Service: {search_service}")
print("   This may take 2-5 minutes...")

search_client = SearchManagementClient(credential, subscription_id)

search_params = SearchService(
    location=location,
    sku=Sku(name='basic'),  # Basic tier supports vector search
    replica_count=1,
    partition_count=1,
    hosting_mode='default',
    identity=Identity(type='SystemAssigned'),  # Enable managed identity
    tags={'project': 'graphrag-lab'}
)

poller = search_client.services.begin_create_or_update(
    resource_group, search_service, search_params
)
search_result = poller.result()
print(f"✅ Search Service created: {search_result.name}")
print(f"   SKU: {search_result.sku.name}")
print(f"   Endpoint: https://{search_service}.search.windows.net")

---

## Step 7: Create Azure OpenAI Service

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Account, Sku as CogSku, AccountProperties, Identity as CogIdentity

openai_account = os.environ['OPENAI_ACCOUNT_NAME']

print(f"🤖 Creating Azure OpenAI Service: {openai_account}")

cognitive_client = CognitiveServicesManagementClient(credential, subscription_id)

openai_params = Account(
    location=location,
    kind='OpenAI',
    sku=CogSku(name='S0'),
    properties=AccountProperties(
        custom_sub_domain_name=openai_account,
        public_network_access='Enabled'
    ),
    identity=CogIdentity(type='SystemAssigned'),
    tags={'project': 'graphrag-lab'}
)

poller = cognitive_client.accounts.begin_create(
    resource_group, openai_account, openai_params
)
openai_result = poller.result()
print(f"✅ OpenAI Service created: {openai_result.name}")
print(f"   Endpoint: {openai_result.properties.endpoint}")

In [ ]:
# Deploy embedding model
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, DeploymentScaleSettings

embedding_deployment = os.environ['EMBEDDING_DEPLOYMENT_NAME']

print(f"📦 Deploying embedding model: {embedding_deployment}")

deployment_params = Deployment(
    properties=DeploymentProperties(
        model=DeploymentModel(
            format='OpenAI',
            name='text-embedding-3-large',
            version='1'
        ),
        scale_settings=DeploymentScaleSettings(
            scale_type='Standard',
            capacity=120  # Tokens per minute (thousands)
        )
    )
)

try:
    poller = cognitive_client.deployments.begin_create_or_update(
        resource_group, openai_account, embedding_deployment, deployment_params
    )
    deployment_result = poller.result()
    print(f"✅ Model deployed: {deployment_result.name}")
except Exception as e:
    print(f"⚠️ Model deployment issue: {e}")
    print("   The model may not be available in this region.")
    print("   Try changing AZURE_LOCATION to 'eastus' or 'westeurope'")

---

## Step 8: Configure RBAC Permissions

Grant the Search service access to OpenAI for embedding generation.

In [ ]:
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
import uuid

print("🔐 Configuring RBAC permissions...")

auth_client = AuthorizationManagementClient(credential, subscription_id)

# Get Search service managed identity
search_service_info = search_client.services.get(resource_group, search_service)
search_principal_id = search_service_info.identity.principal_id

# Get OpenAI resource ID
openai_info = cognitive_client.accounts.get(resource_group, openai_account)
openai_resource_id = openai_info.id

# Cognitive Services OpenAI User role
role_definition_id = f"/subscriptions/{subscription_id}/providers/Microsoft.Authorization/roleDefinitions/5e0bd9bd-7b93-4f28-af87-19fc36ad61bd"

try:
    role_assignment = auth_client.role_assignments.create(
        scope=openai_resource_id,
        role_assignment_name=str(uuid.uuid4()),
        parameters=RoleAssignmentCreateParameters(
            role_definition_id=role_definition_id,
            principal_id=search_principal_id,
            principal_type='ServicePrincipal'
        )
    )
    print(f"✅ RBAC: Search service granted OpenAI access")
except Exception as e:
    if 'RoleAssignmentExists' in str(e):
        print("✅ RBAC: Role assignment already exists")
    else:
        print(f"⚠️ RBAC warning: {e}")

---

## Step 9: Create Storage Account (Optional)

For storing documents before ingestion.

In [ ]:
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.storage.models import (
    StorageAccountCreateParameters,
    Sku as StorageSku,
    Kind
)

storage_account = os.environ['STORAGE_ACCOUNT_NAME']

print(f"📦 Creating Storage Account: {storage_account}")

storage_client = StorageManagementClient(credential, subscription_id)

storage_params = StorageAccountCreateParameters(
    location=location,
    sku=StorageSku(name='Standard_LRS'),
    kind=Kind.STORAGE_V2,
    tags={'project': 'graphrag-lab'}
)

poller = storage_client.storage_accounts.begin_create(
    resource_group, storage_account, storage_params
)
storage_result = poller.result()
print(f"✅ Storage Account created: {storage_result.name}")

---

## Step 10: Display Summary & Connection Info

In [ ]:
print("\n" + "=" * 70)
print("🎉 ENVIRONMENT SETUP COMPLETE!")
print("=" * 70)

print("\n📋 Resource Summary:")
print("-" * 50)
print(f"Resource Group:    {resource_group}")
print(f"Location:          {location}")
print(f"")
print(f"Cosmos DB Account: {cosmos_account}")
print(f"  Database:        {cosmos_database}")
print(f"  Graph:           {cosmos_graph}")
print(f"  Endpoint:        wss://{cosmos_account}.gremlin.cosmos.azure.com:443/")
print(f"")
print(f"Search Service:    {search_service}")
print(f"  Endpoint:        https://{search_service}.search.windows.net")
print(f"")
print(f"OpenAI Account:    {openai_account}")
print(f"  Embedding Model: {embedding_deployment}")
print(f"")
print(f"Storage Account:   {storage_account}")

print("\n" + "=" * 70)
print("📝 NEXT STEPS:")
print("=" * 70)
print("")
print("1. Initialize the graph with sample data:")
print("   python scripts/initialize_graph.py")
print("")
print("2. Create the search index:")
print("   python scripts/setup_search_index.py")
print("")
print("3. Create the search indexer:")
print("   python scripts/setup_search_indexer.py")
print("")
print("4. Run a test query:")
print("   python src/custom_query_graphrag.py --query 'azure ai services'")
print("")
print("Or open graphrag_demo.ipynb for an interactive demo!")
print("=" * 70)

---

## Step 11: Save Configuration to .env File

In [ ]:
# Save configuration to .env file for other scripts
env_content = f"""# GraphRAG Lab Configuration
# Generated by environment_setup.ipynb

AZURE_SUBSCRIPTION_ID={os.environ['AZURE_SUBSCRIPTION_ID']}
RESOURCE_GROUP_NAME={os.environ['RESOURCE_GROUP_NAME']}
AZURE_LOCATION={os.environ['AZURE_LOCATION']}

# Cosmos DB
COSMOS_ACCOUNT_NAME={os.environ['COSMOS_ACCOUNT_NAME']}
COSMOS_DATABASE_NAME={os.environ['COSMOS_DATABASE_NAME']}
COSMOS_GRAPH_NAME={os.environ['COSMOS_GRAPH_NAME']}

# Azure AI Search
SEARCH_SERVICE_NAME={os.environ['SEARCH_SERVICE_NAME']}
SEARCH_INDEX_NAME={os.environ['SEARCH_INDEX_NAME']}

# Azure OpenAI
OPENAI_ACCOUNT_NAME={os.environ['OPENAI_ACCOUNT_NAME']}
EMBEDDING_DEPLOYMENT_NAME={os.environ['EMBEDDING_DEPLOYMENT_NAME']}

# Storage
STORAGE_ACCOUNT_NAME={os.environ['STORAGE_ACCOUNT_NAME']}
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ Configuration saved to .env file")
print("   This file will be used by other scripts automatically.")

---

# 🧹 Cleanup Resources

⚠️ **Run this cell ONLY when you want to delete all resources and stop charges!**

In [ ]:
# ⚠️ DANGER ZONE - This will delete ALL resources!
# Uncomment the lines below to execute cleanup

# CONFIRM_DELETE = True  # Set to True to enable deletion

# if CONFIRM_DELETE:
#     print(f"🗑️ Deleting Resource Group: {resource_group}")
#     print("   This will delete ALL resources in the group!")
#     print("   This may take several minutes...")
#     
#     poller = resource_client.resource_groups.begin_delete(resource_group)
#     poller.result()
#     
#     print("✅ Resource Group deleted successfully!")
#     print("   All Azure charges for this lab have stopped.")
# else:
#     print("⏸️ Cleanup skipped. Set CONFIRM_DELETE = True to proceed.")

print("💡 To cleanup, uncomment the code above and run this cell.")

---

# 📝 Troubleshooting

## Common Issues

### Model Not Available in Region
If embedding model deployment fails:
```python
os.environ['AZURE_LOCATION'] = 'eastus'  # or 'westeurope', 'swedencentral'
```
Then re-run from Step 4.

### Quota Exceeded
Request quota increase in Azure Portal or use an existing resource.

### Authentication Failed
Run in terminal:
```bash
az login
az account set --subscription "your-subscription-id"
```

### Resource Name Already Taken
Azure resource names must be globally unique. Add a suffix:
```python
os.environ['COSMOS_ACCOUNT_NAME'] = 'cosmos-graphrag-dev-abc123'
```